# Photometric Redshifts with Uncertainties
In Lecture 12, we used a neural network to predict each galaxy's redshift from its SDSS colors using MSE loss. Today we'll try varying the likelihood, have it predict uncertainties, and validate the network's predictions. 

In [ ]:
import os
import copy
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from scipy import stats

torch.set_default_dtype(torch.float64)
%matplotlib inline

## load the data

The same 26,700 SDSS galaxies as Lec 12. Run Lec 12 notebook if the cached data is missing. 

In [ ]:
data = np.load('../lec12_nn/data/sdss_photoz.npz')
mag, zspec = data['mag'].astype(float), data['z'].astype(float)

X = np.column_stack([mag[:, :-1] - mag[:, 1:], mag[:, 2]])      # u-g, g-r, r-i, i-z, r
names = ['u-g', 'g-r', 'r-i', 'i-z', 'r']
n = len(zspec)
perm = np.random.default_rng(12).permutation(n)
i_tr, i_va, i_te = perm[:int(0.8 * n)], perm[int(0.8 * n):int(0.9 * n)], perm[int(0.9 * n):]

x_mu, x_sd = X[i_tr].mean(0), X[i_tr].std(0)
z_mu, z_sd = zspec[i_tr].mean(), zspec[i_tr].std()
Xs = torch.tensor((X - x_mu) / x_sd)                 # standardized inputs
ys = torch.tensor((zspec - z_mu) / z_sd)             # standardized target
to_z = lambda y: y * z_sd + z_mu                     # back to redshift
z_te = zspec[i_te]
print(f'train {len(i_tr)}, validation {len(i_va)}, test {len(i_te)};  z_sd = {z_sd:.4f}')


def photoz_metrics(zphot, ztrue):
    ''' photo-z metrics: sigma_NMAD, outlier fraction (|dz|/(1+z) > 0.15) and mean bias of dz = (zphot - ztrue)/(1 + ztrue).
    '''
    dz = (zphot - ztrue) / (1 + ztrue)
    return 1.4826 * np.median(np.abs(dz - np.median(dz))), np.mean(np.abs(dz) > 0.15), np.mean(dz)

Here's the photo-z MLP 

In [ ]:
class PhotozMLP(nn.Module):
    ''' Fully connected ReLU network. sizes = [n_in, hidden_1, ..., hidden_k, n_out].
    '''
    def __init__(self, sizes):
        super().__init__()
        layers = []
        for n_in, n_out in zip(sizes[:-2], sizes[1:-1]):
            layers += [nn.Linear(n_in, n_out), nn.ReLU()]
        layers.append(nn.Linear(sizes[-2], sizes[-1]))
        self.net = nn.Sequential(*layers)
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                # initialize using Kaiming He's initialization
                nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        return self.net(x)            

Same training function as before but now it takes `loss_fn` as input. 

In [ ]:
def fit(model, loss_fn, train=i_tr, val=i_va, epochs=500, lr=1e-3, patience=10, batch=256, seed=1):
    ''' Adam + minibatches + early stopping on the validation loss. Leaves the best weights loaded.
    '''
    # load training data
    loader = DataLoader(TensorDataset(Xs[train], ys[train]), batch_size=batch, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))

    # Adam optimizer
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    
    best_loss, best_epoch, history = np.inf, 0, []
    for epoch in range(epochs):
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            
        with torch.no_grad():
            history.append(loss_fn(model(Xs[val]), ys[val]).item())
            
        if history[-1] < best_loss:
            best_loss, best_epoch, best_state = history[-1], epoch, copy.deepcopy(model.state_dict())
        elif epoch - best_epoch >= patience:
            break
    model.load_state_dict(best_state)
    return np.array(history), best_epoch

1. Train a neural network with [64, 64, 64, 64] hidden layers using the Gaussian likelihood ($\sigma$ fixed). Plot the loss curves. Report the photo-$z$ metrics. Save `z_mse` by running neural network on the test data. 

2. Plot `z_mse` against the true redshift `z_te`. Overplot, mean predicted photo-z in bins of true redshift. Also plot the redshift distribution of training data. Anything interesting? 

3. Instead of Gaussian loss, implement Laplace loss and train a neural network with the same architecture. Plot the loss curves. report the photo-z metrics. How does it compare to the Gaussian loss? Discuss and explain any differences. 

4. Now let's also predict the uncertainties as well. Give the network a second output, $\log\sigma$, and let it learn how uncertain each galaxy is
(*heteroscedastic* regression).

The Gaussian negative log-likelihood per galaxy is
$$-\log p(y \mid \mu, \sigma) = \log\sigma + \frac{(y - \mu)^2}{2\sigma^2} + \tfrac12\log 2\pi .$$

*Hint: In this case, $\mu$ and $\log \sigma$ are outputs of the neural network*

Train 5 multiple of these NN with the same architecture but different `torch.manual_seed(seed)` (seed = 0 - 4) and examine their training and photo-z metrics. How do they compare? Anything unusual? Discuss what could be causing this.

5. Train 5 networks again, this time with a heavier tailed likelihood (Student-$t$)

$$\log s + \frac{\nu + 1}{2}\log\Bigl(1 + \frac{(y - \mu)^2}{\nu s^2}\Bigr)$$

with $\nu = 3$ fixed. How does the training and photo-z metrics compare across the 5 seeds? 

(Optional): Pick the best network with student-t loss. Where is this network unsure in color-magnitude space? 

6. Let's validate the uncertainties using PIT and coverage. Pick the best network with student-t loss. Use `pit_t` function to calculate the PIT and plot the PIT histogram of the test data. Assess the uncertainties.

(Optional): also calculate the coverage for 68% and 95%

In [ ]:
def pit_t(mu, sig, z, nu=3):
    return stats.t.cdf((z - mu) / sig, nu)

